In [1]:
import requests
import datetime
import ujson as json
import adodbapi
import polars as pl
from loguru import logger
#import pandas as pd
            
password = r"St0ck%%Book"
cons_str = f"Provider=Microsoft.SQLSERVER.CE.OLEDB.3.5;Data Source=./data/StockBook.sdf;SSCE:Database Password={password};Mode=ReadWrite|Share Deny None;"
connection = adodbapi.connect(cons_str,)
cursor = connection.cursor()

In [2]:
# 含今天以前的最近交易日

sql = """
SELECT AccountStocks.*, Stock.ClosingPrice FROM AccountStocks LEFT JOIN Stock ON AccountStocks.StockNo = Stock.StockNo ORDER BY AccountNo, TradingDay, TradingNo
"""
cursor.execute(sql)
# print(cursor.description)
sql_data = cursor.fetchall()
list_of_dicts = [dict(zip([column[0] for column in cursor.description], row)) for row in sql_data]
df = pl.DataFrame(data=list_of_dicts, infer_schema_length=10000)

In [3]:
df = (
    df.fill_null(0)
    .with_columns([
        (pl.col("Quantity")*pl.col("UnitPrice")).alias("CostValue"),
        (pl.col("Quantity")*pl.col("ClosingPrice")).alias("MarketValue"),
        pl.col("ClosingPrice").alias("MarketPrice"),
        ])
    .with_columns([
        pl.when(pl.col("HoldingType")==1).then((pl.col("CostValue")*1.001425).floor().round(0))
        .when(pl.col("HoldingType")==2).then(((pl.col("CostValue")*1.001425).floor()-pl.col("Margin")+pl.col("Collateral")).round(0))
        .when(pl.col("HoldingType")==3).then((pl.col("Margin"))).otherwise(0).alias("Cost"),
        
        pl.when(pl.col("HoldingType")==1).then( (pl.col("MarketValue")- (pl.col("MarketValue")*0.001425).floor()- pl.col("MarketValue")*0.003).round(0))
        .when(pl.col("HoldingType")==2).then((pl.col("MarketValue")- (pl.col("MarketValue")*0.001425).floor() - pl.col("MarketValue")*0.003-pl.col("Margin")+pl.col("Collateral")).round(0))
        .when(pl.col("HoldingType")==3).then((pl.col("Collateral")+pl.col("Margin")-pl.col("MarketValue")-(pl.col("MarketValue")*0.001425).floor())).otherwise(0).alias("RealizedValue"),
    ])
    .with_columns([
        (pl.col("RealizedValue")-pl.col("Cost")).round(0).alias("GainLoss"),
        pl.when(pl.col("Cost")==0).then(0).otherwise((pl.col("RealizedValue")-pl.col("Cost"))/pl.col("Cost")).round(4).fill_null(0).alias("GainLossPercent"),

        pl.when(pl.col("HoldingType")==1).then(0)
        .when(pl.col("HoldingType")==2).then(pl.col("Collateral")+pl.col("MarketValue")-(pl.col("MarketValue")*0.001425).floor()- pl.col("MarketValue")*0.003)
        .when(pl.col("HoldingType")==3).then( (pl.col("Collateral")+pl.col("Margin"))/ (pl.col("MarketValue")- (pl.col("MarketValue")*0.001425).floor())).otherwise(0).round(4).alias("MaintenanceRatio"),
    ]).select(
            [
                "Cost",
                "MarketPrice",
                "MarketValue",
                "MaintenanceRatio",
                "GainLoss",
                "GainLossPercent",
                "AccountNo",
                "TradingDay",
                "TradingNo",
            ]
))

In [5]:
sql = """UPDATE AccountStocks SET MarketPrice = 0.0, MarketValue = 0, MaintenanceRatio = 0.0, GainLoss = -1031467, GainLossPercent = -1.0 WHERE AccountNo = '1234' AND TradingDay = '2017-02-17 00:00:00' AND TradingNo = '1'"""
cursor.execute(sql)

In [10]:
# for idx,one in enumerate(df.to_dicts()):
# idx =0
# one = {'MarketPrice': 0.0, 'MarketValue': 0.0, 'MaintenanceRatio': 0.0, 'GainLoss': -1031467.0, 'GainLossPercent': -1.0, 'AccountNo': '1234', 'TradingDay': datetime.datetime(2017, 2, 17, 0, 0), 'TradingNo': 1}
# update_sql = f"""UPDATE AccountStocks SET MarketPrice = {one['MarketPrice']}, MarketValue = {int(one['MarketValue'])}, MaintenanceRatio = {one['MaintenanceRatio']}, GainLoss = {int(one['GainLoss'])}, GainLossPercent = {one['GainLossPercent']} WHERE AccountNo = '{one['AccountNo']}' AND TradingDay = '{one['TradingDay']}' AND TradingNo = '{one['TradingNo']}'"""
update_sql = """UPDATE AccountStocks SET MarketPrice = 19.45, MarketValue = 1478200, MaintenanceRatio = 0.0, GainLoss = 785924, GainLossPercent = 1.1461 WHERE AccountNo = '2384' AND TradingDay = '2020-06-10 00:00:00' AND TradingNo = '10'"""
try:
    cursor.execute(update_sql)
except Exception as e:
    logger.warning(f" 更新庫存明細失敗 {one} [{e}]")



In [ ]:
sql= """select * from Account"""
cursor.execute(sql)
list_of_dicts = [dict(zip([column[0] for column in cursor.description], row)) for row in sql_data]
pl.DataFrame(data=list_of_dicts, infer_schema_length=10000)

sql= """select * from LoanBalance"""
cursor.execute(sql)
list_of_dicts = [dict(zip([column[0] for column in cursor.description], row)) for row in sql_data]
pl.DataFrame(data=list_of_dicts, infer_schema_length=10000)


In [11]:
a = ["""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 330000, MaintenanceRatio = 0.0, GainLoss = 229600, GainLossPercent = 2.3206 WHERE AccountNo = '2890' AND TradingDay = '2019-08-06 00:00:00' AND TradingNo = '6'"""
,"""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 330000, MaintenanceRatio = 0.0, GainLoss = 229900, GainLossPercent = 2.3307 WHERE AccountNo = '2890' AND TradingDay = '2019-08-06 00:00:00' AND TradingNo = '7'"""
,"""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 1089000, MaintenanceRatio = 0.0, GainLoss = 748755, GainLossPercent = 2.2322 WHERE AccountNo = '2890' AND TradingDay = '2019-08-07 00:00:00' AND TradingNo = '3'"""
,"""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 165000, MaintenanceRatio = 0.0, GainLoss = 112947, GainLossPercent = 2.2007 WHERE AccountNo = '2890' AND TradingDay = '2019-08-07 00:00:00' AND TradingNo = '4'"""
,"""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 264000, MaintenanceRatio = 0.0, GainLoss = 179915, GainLossPercent = 2.1698 WHERE AccountNo = '2890' AND TradingDay = '2019-08-07 00:00:00' AND TradingNo = '5'"""
,"""UPDATE AccountStocks SET MarketPrice = 33.0, MarketValue = 33000, MaintenanceRatio = 0.0, GainLoss = 22440, GainLossPercent = 2.1548 WHERE AccountNo = '2890' AND TradingDay = '2019-08-07 00:00:00' AND TradingNo = '6'"""]
for i in a:
    cursor.execute(i)

In [ ]:
def list_pad_dates(cursor,start_date):
    end_date = datetime.date.today()
    cursor.execute(f"SELECT Day FROM Schedule WHERE day>'{start_date}' AND day<='{end_date}' AND IsClosed=0 ORDER BY Day")
    return [i._getValue(0).date() for i in  cursor.fetchall()]


def latest_loan_data(cursor):
    sql = """SELECT * FROM LoanBalance"""
    cursor.execute(sql)
    sql_data = cursor.fetchall()
    columns = [column[0] for column in cursor.description]
    # print(columns)
    df = pl.DataFrame(data=list(sql_data.ado_results), schema=columns)
    df = df.sort(["AccountNo","AccountDay"])
    return df.groupby("AccountNo").agg(pl.col("AccountDay").last()
                                ,pl.col("Loan").last()
                                ,pl.col("InterestDate").last()
                                ,pl.col("InterestDays").last()
                                ,pl.col("LoanInterest").last()
                                )


def latest_account(cursor):
    sql = """SELECT AccountNo,LoanRate FROM Account WHERE MarginType=2 AND IsDeleted=0 GROUP BY Account.AccountNo, Account.LoanRate"""
    cursor.execute(sql)
    sql_data = cursor.fetchall()
    list_of_dicts = [dict(zip([column[0] for column in cursor.description], row)) for row in sql_data]
    return pl.DataFrame(data=list_of_dicts)

def current_account_loan(cursor):
    df = latest_account(cursor)
    df = df.join(latest_loan_data(cursor),on="AccountNo",how="left")
    return df

def get_interest_date_and_days(all_open_date,open_date=datetime.date.today()):    
    interest_date = [day for day in all_open_date if day >= open_date][2]
    next_open_date = [day for day in all_open_date if day >= open_date][3]
    return open_date,interest_date, (next_open_date - interest_date).days

def get_pading_loan_data(cursor):
    cursor.execute(f"SELECT Day FROM Schedule WHERE IsClosed=0 and day<='{str(datetime.date.today()+datetime.timedelta(days=30))}' ORDER BY Day")
    all_open_date = [i._getValue(0).date() for i in cursor.fetchall()]
    get_interest_date_and_days(all_open_date)
    df = current_account_loan(cursor)

    pad_data = []
    for one_df in df.to_dicts():
        if one_df['AccountDay']:
            relevant_days = [day for day in list_pad_dates(cursor,one_df['AccountDay']) if day > one_df['AccountDay'].date()]
            for date in relevant_days:
                loan_rate = one_df['LoanRate']
                loan = int(one_df['Loan'])
                open_date,interest_date,interest_days = get_interest_date_and_days(all_open_date,date)
                load_interest = 0 if loan <= 0 else round(loan * loan_rate * interest_days)
                new_record = (one_df['AccountNo'], open_date , loan, interest_date , interest_days, load_interest)
                pad_data.append(new_record)
    return pad_data

In [ ]:
pading_loan_data =  get_pading_loan_data(cursor)

In [ ]:
pading_loan_data[:5]

In [ ]:
if pading_loan_data:
    logger.info(f"更新日期 從 {pading_loan_data[0][1]} -> {pading_loan_data[-1][1]} 共 {len(pading_loan_data)} 筆")
    insert_sql = """INSERT INTO LoanBalance  (accountno, accountday, loan,interestdate, interestdays, loaninterest) VALUES (?,?, ?,?, ?, ? )"""
    try:
        cursor.executemany(insert_sql,*pading_loan_data[:1])
    except Exception as e:
        logger.warning(f"帳戶信用更新 [{e}]")
        logger.warning(f"{insert_sql}")

else:
    logger.info(f"無可更新資料")


In [ ]:
cursor.execute(f"SELECT TOP (1) day FROM Schedule WHERE day<='{str(datetime.date.today())}' AND IsClosed=0 ORDER BY Day DESC")
result = cursor.fetchall()
result[0][0]


In [ ]:
#SELECT TOP (2) Day FROM {0} WHERE Day > @Day AND IsClosed = @IsClosed ORDER BY Day
cursor.execute(f"SELECT TOP (1) Day FROM Schedule WHERE day<='{str(datetime.date.today())}' AND IsClosed=0 ORDER BY Day desc")
result = cursor.fetchall()
result[0][0]

In [ ]:
#SELECT TOP (2) Day FROM {0} WHERE Day > @Day AND IsClosed = @IsClosed ORDER BY Day
cursor.execute(f"SELECT TOP (2) Day FROM Schedule WHERE day>'{str(datetime.date.today())}' AND IsClosed=0 ORDER BY Day")
sdf = cursor.fetchall()
logger.info(sdf[1][0])
    

In [ ]:
def get_open_day(open_date=None):
    # Implement your logic here
    if open_date==None:
        open_date = datetime.date.today()
    
    cursor.execute(f"SELECT TOP (1) Day FROM Schedule WHERE day<='{open_date}' AND IsClosed=0 ORDER BY Day desc")
    result = cursor.fetchall()
    open_day = result[0][0]

    cursor.execute(f"SELECT TOP (3) Day FROM Schedule WHERE day>'{open_date}' AND IsClosed=0 ORDER BY Day")
    result = cursor.fetchall()
    interest_begin = result[1][0]
    interest_end = result[2][0]
    interest_days = (interest_end - interest_begin).days
    info_dates = {"open_date":open_day,"interest_begin":interest_begin,"interest_end":result[2][0],"interest_days":interest_days}
    # logger.info(f"開帳日期 {info_dates['open_date'].date()}\n起息日 {info_dates['interest_begin'].date()}\n結利日(不含) {info_dates['interest_end'].date()}\n利息天數 {info_dates['interest_days']} 天")
    return info_dates

get_open_day("2022/12/29")


In [ ]:
def list_pad_dates(start_date):
    end_date = datetime.date.today()
    cursor.execute(f"SELECT Day FROM Schedule WHERE day>'{start_date}' AND day<='{end_date}' AND IsClosed=0 ORDER BY Day")
    return [i._getValue(0).date() for i in  cursor.fetchall()]


def latest_loan_data():
    sql = """SELECT * FROM LoanBalance"""
    cursor.execute(sql)
    sql_data = cursor.fetchall()
    columns = [column[0] for column in cursor.description]
    # print(columns)
    df = pl.DataFrame(data=list(sql_data.ado_results), schema=columns)
    df = df.sort(["AccountNo","AccountDay"])
    return df.groupby("AccountNo").agg(pl.col("AccountDay").last()
                                ,pl.col("Loan").last()
                                ,pl.col("InterestDate").last()
                                ,pl.col("InterestDays").last()
                                ,pl.col("LoanInterest").last()
                                )

def latest_account():
    # cursor = self.cursor
    sql = """SELECT AccountNo,LoanRate FROM Account WHERE MarginType=2 AND IsDeleted=0 GROUP BY Account.AccountNo, Account.LoanRate"""
    cursor.execute(sql)
    sql_data = cursor.fetchall()
    list_of_dicts = [dict(zip([column[0] for column in cursor.description], row)) for row in sql_data]
    return pl.DataFrame(data=list_of_dicts)
    
df = latest_account()
df = df.join(latest_loan_data(),on="AccountNo",how="left")
df.head()

In [ ]:
cursor.execute(f"SELECT Day FROM Schedule WHERE IsClosed=0 and day<='{str(datetime.date.today()+datetime.timedelta(days=30))}' ORDER BY Day")
all_open_date = [i._getValue(0).date() for i in cursor.fetchall()]

def get_interest_date_and_days(all_open_date,open_date=datetime.date.today()):    
    interest_date = [day for day in all_open_date if day >= open_date][2]
    next_open_date = [day for day in all_open_date if day >= open_date][3]
    return open_date,interest_date, (next_open_date - interest_date).days
get_interest_date_and_days(all_open_date)

pad_data = []
for one_df in df.to_dicts():
    if one_df['AccountDay']:
        relevant_days = [day for day in list_pad_dates(one_df['AccountDay']) if day > one_df['AccountDay'].date()]
        for date in relevant_days:
            loan_rate = one_df['LoanRate']
            loan = int(one_df['Loan'])
            open_date,interest_date,interest_days = get_interest_date_and_days(all_open_date,date)
            load_interest = 0 if loan <= 0 else round(loan * loan_rate * interest_days)
            new_record = (one_df['AccountNo'], open_date , loan, interest_date , interest_days, load_interest)
            pad_data.append(new_record)
pad_data

In [ ]:
cursor.execute(f"SELECT Day FROM Schedule WHERE IsClosed=0 and day<='{str(datetime.date.today()+datetime.timedelta(days=30))}' ORDER BY Day")
all_open_date = [i._getValue(0).date() for i in cursor.fetchall()]

def get_interest_date_and_days(all_open_date,open_date=datetime.date.today()):    
    interest_date = [day for day in all_open_date if day >= open_date][2]
    next_open_date = [day for day in all_open_date if day >= open_date][3]
    return open_date,interest_date, (next_open_date - interest_date).days
get_interest_date_and_days(all_open_date)


In [ ]:
pad_data[:5]

In [ ]:
append_data = []
for loan_info in all_loan_account.to_dicts():
    account = loan_info.get("AccountNo")
    last_date = loan_info.get("AccountDay")
    

In [ ]:
result = df.join(df1, on="AccountNo", how="left")

